# Triaxial permeation, TWO-PISTON — the one permeation run

Analysis for `triaxial_permeation_two_pist.lmp`: a laterally periodic gel slab between a **feed** reservoir held
at `P_target + dP` and a **permeate** reservoir held at `P_target`, each closed by an NPT-piston (Marioni et al.,
J. Membr. Sci. 738 (2026) 124837, Eq. 3; the paper's Fig. 2B rotated by 90°).  There is **no sweep**: the whole
production run is one continuous constant-pressure drive at one `dP`, so every field is a **time evolution** over
the production frames (colour = timestep, cividis; bold black = final).  The Phase 1.5 **zero-flux reference**
(both pistons at `P_target`) is the dashed baseline.

Sections: profile evolutions (total / partial / network stress, density) with the reference baseline; reservoir
pressures **measured on the pistons vs applied**; `Q_perm(t)` from the permeate-piston displacement with its
steady-state block-bootstrap plateau; the permeability `k = Q_perm L /(A dP)` with a CI; the bead-count
cross-check.  All code is in `scripts/lib/triaxial.py` (`load_permeation`, `fig_perm_*`).  Notes at the end
(2026-09-16).

## Files (synced by the sync cell)

Into `flow_data_local/permeation/<RUN_ID>/` (cluster `output_files/…`, names `<name>_<DATANAME>_<INTERACTION>_<NSTEPS>`):

| file | content |
|---|---|
| `sigma{zz,xx,yy}_{polymer,solvent}[_ref]_…` | group partial stress profiles (kinetic term included) |
| `solvent_density_z[_ref]_…` | solvent number / mass density profiles |
| `piston_position_…`, `piston_velocity_…`, `piston_force_…`, `piston_force_avg[_ref]_…` | `[feed \| perm]` columns |
| `piston_pressure_…` | `P_feed`, `P_perm` measured (`F_fluid/(lx ly)`) and applied |
| `permeation_…` | block-averaged `z_feed z_perm F_feed F_perm P_feed P_perm Q_perm N_permeate` |
| `permeate_count_…` | bead count below the support (cross-check) |
| `pressure_feed_…`, `pressure_permeate_…` | reservoir virial pressures + densities |
| `strain_zz_…`, `gel_dimensions_{rg,bb}_…`, `box_dimensions_…`, `polymer_com_…`, `stress_aniso_…` | geometry / diagnostics |
| `disp_z_polymer_…` | polymer displacement profile (not used here) |

Into `flow_data_local/traj_files.nosync/`: `traj_ref_…` (box header + wall planes), `traj_stress_…`.

## 1 · Setup and computation

In [ ]:
import sys, importlib
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

LIB = Path('lib').resolve()            # scripts/lib: triaxial.py (all analysis code) + volfrac.py
if str(LIB) not in sys.path:
    sys.path.insert(0, str(LIB))
import triaxial as tri
tri = importlib.reload(tri)            # pick up edits to lib/triaxial.py without a kernel restart
tri.setup_style()
print('analysis code: ', LIB / 'triaxial.py')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  CONFIG -- the only cell to edit when switching runs
# ══════════════════════════════════════════════════════════════════════════
cfg = tri.Config(
    DATANAME    = "final_config_slab_support_periodic_5beads_tall_rho04_new_1.0_1.0_14000002_two_pist",
    INTERACTION = "1.0_1.0",          # epsSS_epsSP
    NSTEPS      = None,           # production steps = the <steps> tag; None resolves it from the files after the sync
    RUN_ID      = "two_pist_perm_1",  # local folder under flow_data_local/{permeation,plots}
    mode        = "permeation",   # NO levels: one continuous constant-dP drive
    two_pist    = True,           # Expanse folder triaxial_permeation_two_pist
    DP_PISTON   = None,           # applied dP; None -> read from piston_pressure (P_feed_app - P_perm_app)
    plateau_frac      = 0.25,     # trailing fraction used for the "steady" profile means
    plateau_frac_auto = 0.45,     # longest candidate window for the Q_perm steady-state plateau (drift test)
    P_BARO      = 1.5,            # P_target (normalises the total-stress panels)
)

In [ ]:
SYNC, FORCE_SYNC = True, False
if SYNC:
    tri.sync_from_expanse(cfg, force=FORCE_SYNC)

In [ ]:
# R -- the zero-flux reference (Phase 1.5; both pistons at P_target): geometry, reference profiles, wall planes
# P -- the production run: stress / density evolutions, pistons, Q_perm(t), permeability
R = tri.load_reference(cfg)
P = tri.load_permeation(cfg, R)
tri.print_perm_summary(cfg, R, P)

## 2 · Figures

In [ ]:
# 1 · Wet pistons: (a) displacement of the feed and permeate pistons; (b) P measured (F_fluid/A) vs applied, reservoir virial dotted
tri.fig_perm_pistons(cfg, R, P);

In [ ]:
# 2 · Total stress / P_bath evolution σ^t_zz, σ^t_xx, σ^t_yy -- zero-flux reference dashed -> drive (cividis) -> final (bold)
tri.fig_total_stress(cfg, R, P);

In [ ]:
# 3 · Solvent and polymer partial σ_zz evolutions with the total superimposed
tri.fig_partial_stress(cfg, R, P);

In [ ]:
# 4 · Network stress evolution σ' = σ^t − p_pore (feed-reservoir baseline)
tri.fig_network_stress(cfg, R, P);

In [ ]:
# 5 · Solvent mass-density evolution with the zero-flux reference
tri.fig_perm_density(cfg, R, P);

In [ ]:
# 6 · Q_perm(t) from the permeate-piston displacement: steady window, block-bootstrap mean, z_perm fit; bead-count cross-check
tri.fig_perm_flux(cfg, R, P);

In [ ]:
# 7 · Permeability k = Q_perm L/(A dP): applied dP, measured dP (wet pistons), bead count -- with CIs
tri.fig_perm_permeability(cfg, R, P);

In [ ]:
# 8 · Thermodynamic pressure P_th = −⅓ tr(σ^t) evolution (positive under compression; dotted = P_bath)
tri.fig_thermo_pressure(cfg, R, P);

In [ ]:
# 9 · Osmotic pressure Π = −⅓ tr(σ′): zero-flux reference and final (steady-flow) state, feed-reservoir baseline.
#     Under flow p_pore is not uniform across the gel, so the final curve's slope is mostly the pore-pressure drop
tri.fig_osmotic_pressure(cfg, R, P);


## Notes

* **Flux**.  The primary flux is the permeate-piston displacement, `Q_perm = A · dz_perm/dt`, block-averaged by
  the deck over each `volume_freq` window (`permeation_….dat`, column `Q_perm`); positive = flow into the
  permeate reservoir (the piston retreats).  `tri.plateau_window` picks the longest drift-free trailing window
  (both halves agree within their block-bootstrap CIs) and gives the steady mean with a CI; a linear fit of
  `z_perm(t)` over the same window is the second estimate.  The **bead count** (`permeate_count`, solvent that
  crossed below the support, the one-piston method) divided by the reference bulk density `ρ_s,0` is the
  cross-check.
* **Permeability**.  `k = Q_perm L /(A ΔP)` with `L` the Rg-based gel thickness over the steady window and
  `A = l_x l_y`; reported with the **applied** `ΔP` (= `dp_piston`) and with the `ΔP` **measured** on the
  pistons (`F_fluid/A`, block-bootstrapped).  The CI combines the relative CI of `Q` and of `ΔP_meas` in
  quadrature.  In LJ units `k` is `σ⁵/(ε τ)`; it equals the `κ = k/η` of the compression notebooks (η = 1 in
  these units is NOT assumed — compare `κ = D_c/M` from the compression sweep with this `k` to test Darcy
  consistency).
* **Pressures**.  `P_feed_meas`, `P_perm_meas` are the pair forces of the mobile atoms on each sheet over
  `l_x l_y` (compute group/group; NOT `reduce sum fz`, which the aveforce fix zeroes).  They must average to the
  applied values; the reservoir virial pressures (`pressure_feed`, `pressure_permeate`, dotted in figure 1b) are
  the independent check.  Thermo `press` is meaningless in this geometry (vacuum margins in the box).
* **Profiles**.  z-binning covers the full box (bins in the vacuum read 0).  The pore baseline for the network
  stress is the feed-reservoir interior; `pore_perm` (the permeate side) is also stored in `P['stress'][comp]`.
* **Halts**.  The deck stops when the feed reservoir thins to `feed_halt_thick` or a piston face comes within
  2 σ of a box face; the run then ends cleanly (files are complete).